# MNLI Benchmark Analysis — Groups 1 & 2

Fetches all MNLI runs from W&B across heterogeneity conditions,
computes derived metrics, prints per-group and cross-alpha summaries,
and saves to `results/mnli_all.pkl`.

**R\* criterion:** patience-based early stopping — R\* is the first round
(after warmup) where the running best accuracy has not improved by more
than δ for P consecutive rounds.

In [1]:
import os
import pickle
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import wandb

warnings.filterwarnings("ignore")

## Configuration

In [2]:
STRATEGIES = ["random", "fedcs", "tifl", "oort"]
ALPHAS = [0.5, 2.0, "iid"]  # ordered: high non-IID → moderate → IID
ALPHA_LABELS = {0.5: "α=0.5", 2.0: "α=2", "iid": "IID"}

# R* early stopping
R_MIN    = 20     # warmup: early stopping not active before this round
PATIENCE = 10     # consecutive rounds without improvement to declare convergence
DELTA    = 0.005  # accuracy: min improvement threshold (same scale as acc metric)
DELTA_LOSS = 0.001  # loss: min improvement threshold (loss is more sensitive)

# Experiment constants
TARGET_ACC = 0.80
R_MAX = 100
N_CLIENTS = 300

OUTPUT_PATH = "results/mnli_all.pkl"
os.makedirs("results", exist_ok=True)

In [3]:
ENTITY  = "ilham-abdillah-alhamdi-universitas-indonesia"
PROJECT = "u-flora-experiments"

# ── Group names per alpha condition ──────────────────────────────────────
GROUPS = {
    2.0:   "final-group1-mnli",
    0.5:   "final-group2-mnli-a-0.5",
    "iid": "final-group2-mnli-iid",
}

def _parse_alpha_key(partition: str, raw_alpha) -> float | str:
    """Determine the alpha key for RUNS dict from partition strategy and raw alpha."""
    if partition == "iid":
        return "iid"
    # Dirichlet: normalize to canonical float key
    val = float(raw_alpha)
    for canonical in [0.5, 2.0]:
        if abs(val - canonical) < 0.01:
            return canonical
    return val


In [4]:
import wandb
api = wandb.Api(timeout=60)

RUNS = {}

print("Fetching run IDs from W&B groups...\n")

for alpha_key, group_name in GROUPS.items():
    print(f"  Group '{group_name}' (alpha={alpha_key}):")
    group_runs = api.runs(f"{ENTITY}/{PROJECT}", filters={"group": group_name})

    found = []
    for run in group_runs:
        try:
            partition  = run.config["dataset"]["partition"]["strategy"]
            raw_alpha  = run.config["dataset"]["partition"]["dirichlet"]["alpha"]
            strategy   = run.config["strategy"]["name"].lower().strip()
            seed       = int(run.config["seed"])
            parsed_key = _parse_alpha_key(partition, raw_alpha)
        except (KeyError, TypeError, ValueError) as e:
            print(f"    ✗ Run {run.id} ({run.name}): could not parse config — {e}")
            continue

        # Only keep runs that belong to the expected alpha_key for this group
        if parsed_key != alpha_key:
            print(f"    ⚠  Skipping {run.id}: parsed alpha={parsed_key} != expected {alpha_key}")
            continue

        if alpha_key not in RUNS:
            RUNS[alpha_key] = {}
        if strategy not in RUNS[alpha_key]:
            RUNS[alpha_key][strategy] = {}

        if seed in RUNS[alpha_key].get(strategy, {}):
            print(f"    ⚠  Duplicate: {strategy} seed={seed} — "
                  f"keeping {RUNS[alpha_key][strategy][seed]}, ignoring {run.id}")
        else:
            RUNS[alpha_key][strategy][seed] = run.id
            found.append((strategy, seed, run.id, run.name))

    for strategy, seed, run_id, run_name in sorted(found):
        print(f"    ✓ {strategy:8s}  seed={seed:3d}  {run_id}  ({run_name})")

    # Validate expected seeds are present
    expected_seeds = {2.0: [42, 123, 456], 0.5: [42, 123, 456], "iid": [42]}
    for strat in STRATEGIES:
        expected = expected_seeds.get(alpha_key, [42])
        present  = list(RUNS.get(alpha_key, {}).get(strat, {}).keys())
        missing  = [s for s in expected if s not in present]
        if missing:
            print(f"    ✗ MISSING: {strat} seeds {missing}")
    print()

print("Done. RUNS structure:")
for alpha in RUNS:
    for strat in RUNS.get(alpha, {}):
        seeds = list(RUNS[alpha][strat].keys())
        print(f"  alpha={alpha}  {strat:8s}  seeds={seeds}")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\ASUS\_netrc.


Fetching run IDs from W&B groups...

  Group 'final-group1-mnli' (alpha=2.0):
    ✓ fedcs     seed= 42  r39tsb13  (fedcs-mnli-42-a2-v1-20260506_09:35:22)
    ✓ fedcs     seed=123  qqarz0um  (fedcs-mnli-123-a2-v1-20260508_13:44:56)
    ✓ fedcs     seed=456  3z3lad4v  (fedcs-mnli-456-a2-v1-20260508_22:57:59)
    ✓ oort      seed= 42  g7bgan0t  (oort-mnli-42-a2-v4-20260524_07:14:03)
    ✓ oort      seed=123  1ez5fv2u  (oort-mnli-123-a2-v4-20260524_09:17:01)
    ✓ oort      seed=456  k79yl52h  (oort-mnli-456-a2-v4-20260524_11:17:35)
    ✓ random    seed= 42  ukqx8xyv  (random-mnli-42-a2-v1-20260506_05:21:22)
    ✓ random    seed=123  jhx7m47m  (random-mnli-123-a2-v1-20260508_11:23:50)
    ✓ random    seed=456  teapg762  (random-mnli-456-a2-v1-20260508_20:37:12)
    ✓ tifl      seed= 42  9jtvaulj  (tifl-mnli-42-a2-v2-20260507_20:52:12)
    ✓ tifl      seed=123  i6znx4zl  (tifl-mnli-123-a2-v1-20260508_15:44:19)
    ✓ tifl      seed=456  8m686cay  (tifl-mnli-456-a2-v1-20260509_00:56:00)

  Gr

## W&B Columns & Utility Functions

In [5]:
HISTORY_KEYS = [
    # ── Round-level ────────────────────────────────────────────────────
    "round/server_round",
    "round/wall_clock",
    "round/cumulative_wall_clock",
    "round/duration_mean",
    "round/duration_std",
    "round/num_client_selected",
    "round/num_client_completed",
    # ── Evaluation ────────────────────────────────────────────────────
    "eval/accuracy",
    "eval/loss",
    # ── Fairness ──────────────────────────────────────────────────────
    "fairness/jain_index",
    "fairness/unique_clients_explored",
    "fairness/exploration_ratio",
    # ── Utility signal (for RQ2 mechanism analysis) ───────────────────
    "utility/loss_cv",
    "utility/loss_std",
    "utility/loss_rms_cv",
    "utility/loss_rms_std",
    # ── Oort-specific (will be NaN for non-Oort runs) ─────────────────
    "strategy/oort_preferred_t",
    "strategy/oort_epsilon",
    "strategy/oort_round_threshold",
]

In [6]:
def compute_r_star(series: pd.Series, higher_is_better: bool = False) -> int:
    """Patience-based early stopping. Returns 1-indexed round.

    higher_is_better=False (default): loss mode — fires when loss stops
    decreasing by more than DELTA_LOSS for PATIENCE rounds. Standard practice.

    higher_is_better=True: accuracy mode — fires when accuracy stops
    increasing by more than DELTA for PATIENCE rounds. Kept for reference.
    """
    vals = series.values.astype(float)
    n = len(vals)
    best = -np.inf if higher_is_better else np.inf
    threshold = DELTA if higher_is_better else DELTA_LOSS
    counter = 0

    for r in range(n):
        improved = (
            vals[r] > best + threshold
            if higher_is_better
            else vals[r] < best - threshold
        )
        if improved:
            best = vals[r]
            counter = 0
        else:
            counter += 1
        if r >= R_MIN and counter >= PATIENCE:
            return r + 1

    return R_MAX


def first_crossing(series: pd.Series, threshold: float) -> int | None:
    """1-indexed round of first value >= threshold."""
    crossed = np.where(series.ffill().values >= threshold)[0]
    return int(crossed[0]) + 1 if len(crossed) > 0 else None


def safe_iloc(series: pd.Series, idx_1based: int):
    """0-indexed access with bounds check."""
    i = idx_1based - 1
    if i < 0 or i >= len(series):
        return None
    v = series.iloc[i]
    return None if pd.isna(v) else float(v)


def safe_col(hist: pd.DataFrame, key: str) -> pd.Series:
    """Return column if it exists, else NaN series of same length."""
    if key in hist.columns:
        return hist[key]
    return pd.Series([np.nan] * len(hist), index=hist.index)

## Fetch & Compute Pipeline

In [7]:
def fetch_run(run_id: str) -> tuple[pd.DataFrame, dict]:
    api = wandb.Api(timeout=60)
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")

    hist = run.history(samples=500, pandas=True)

    if "round/server_round" not in hist.columns:
        raise ValueError(f"Run {run_id}: 'round/server_round' column missing")

    hist = hist.dropna(subset=["round/server_round"]).copy()
    hist["round/server_round"] = hist["round/server_round"].astype(int)
    hist = hist.sort_values("round/server_round").reset_index(drop=True)

    summary = dict(run.summary)
    return hist, summary

In [8]:
def compute_metrics(hist: pd.DataFrame, summary: dict, run_id: str) -> dict:

    rounds = hist["round/server_round"]
    acc = hist["eval/accuracy"].ffill()
    loss = safe_col(hist, "eval/loss")
    cum_wc = hist["round/cumulative_wall_clock"]
    wall_clock = hist["round/wall_clock"]
    dur_mean = hist["round/duration_mean"]
    dur_std = hist["round/duration_std"]
    jfi = hist["fairness/jain_index"]
    unique_exp = safe_col(hist, "fairness/unique_clients_explored")
    loss_cv = safe_col(hist, "utility/loss_cv")
    loss_rms_cv = safe_col(hist, "utility/loss_rms_cv")

    # Oort-specific (NaN for non-Oort)
    oort_preferred_t = safe_col(hist, "strategy/oort_preferred_t")
    oort_epsilon = safe_col(hist, "strategy/oort_epsilon")
    oort_round_threshold = safe_col(hist, "strategy/oort_round_threshold")

    # ── Convergence ──────────────────────────────────────────────────
    r_star = min(compute_r_star(acc, higher_is_better=True), R_MAX)
    # RTA still uses accuracy — it's the primary task metric.
    rta = first_crossing(acc, TARGET_ACC)

    # ── Wall-clock efficiency ────────────────────────────────────────
    tta = safe_iloc(cum_wc, rta) if rta else None
    wc_at_r_star = safe_iloc(cum_wc, r_star)
    mean_round_dur = float(dur_mean.iloc[:r_star].mean())

    # ── Accuracy ─────────────────────────────────────────────────────
    acc_at_r_star = safe_iloc(acc, r_star)
    acc_at_r_max = safe_iloc(acc, R_MAX) or float(acc.iloc[-1])
    best_acc = float(summary.get("summary/best_metric", acc.max()))

    # ── Fairness ─────────────────────────────────────────────────────
    jfi_at_r_star = safe_iloc(jfi, r_star)
    jfi_at_r_max = safe_iloc(jfi, R_MAX) or float(jfi.iloc[-1])
    delta_jfi = (jfi_at_r_max - jfi_at_r_star) if jfi_at_r_star is not None else None

    unique_at_r_star = safe_iloc(unique_exp, r_star)
    never_selected_at_r_star = (
        int(N_CLIENTS - unique_at_r_star) if unique_at_r_star is not None else None
    )

    # ── Straggler ────────────────────────────────────────────────────
    strag_ratio = wall_clock / dur_mean.replace(0.0, np.nan)
    strag_overhead = float((strag_ratio.iloc[:r_star] - 1.0).clip(lower=0).mean())

    return {
        # ── Scalars ──────────────────────────────────────────────────
        "run_id": run_id,
        "r_star": r_star,
        "rta": rta,
        "tta": tta,
        "wc_at_r_star": wc_at_r_star,
        "mean_round_dur": mean_round_dur,
        "acc_at_r_star": acc_at_r_star,
        "acc_at_r_max": acc_at_r_max,
        "best_acc": best_acc,
        "jfi_at_r_star": jfi_at_r_star,
        "jfi_at_r_max": jfi_at_r_max,
        "delta_jfi": delta_jfi,
        "unique_at_r_star": unique_at_r_star,
        "never_selected": never_selected_at_r_star,
        "strag_overhead": strag_overhead,
        # ── Series (for plotting notebook) ───────────────────────────
        "rounds": rounds,
        "acc_series": acc,
        "loss_series": loss,
        "cum_wc_series": cum_wc,
        "wall_clock_series": wall_clock,
        "dur_mean_series": dur_mean,
        "dur_std_series": dur_std,
        "strag_ratio_series": strag_ratio,
        "jfi_series": jfi,
        "unique_exp_series": unique_exp,
        "loss_cv_series": loss_cv,
        "loss_rms_cv_series": loss_rms_cv,
        # ── Oort-specific series ─────────────────────────────────────
        "oort_preferred_t_series": oort_preferred_t,
        "oort_epsilon_series": oort_epsilon,
        "oort_round_threshold_series": oort_round_threshold,
    }

## Main: Fetch All Runs

In [9]:
all_results = {}  # {alpha: {strategy: {seed: metrics_dict}}}

for alpha in ALPHAS:
    print(f"\n{'='*70}")
    print(f"  Alpha = {ALPHA_LABELS[alpha]}")
    print(f"{'='*70}")
    all_results[alpha] = {}

    for strategy in STRATEGIES:
        all_results[alpha][strategy] = {}
        seeds = list(RUNS[alpha][strategy].keys())

        for seed in seeds:
            run_id = RUNS[alpha][strategy][seed]

            print(f"  {strategy:6s} seed={seed:3d}  ({run_id}) ...", end=" ", flush=True)
            try:
                hist, summary = fetch_run(run_id)
                metrics = compute_metrics(hist, summary, run_id)
                all_results[alpha][strategy][seed] = metrics
                print(
                    f"OK  R*={metrics['r_star']}  "
                    f"RTA={metrics['rta']}  "
                    f"best={metrics['best_acc']:.4f}"
                )
            except Exception as e:
                print(f"FAILED: {e}")
                all_results[alpha][strategy][seed] = None


  Alpha = α=0.5
  random seed= 42  (kwo6g2hd) ... OK  R*=59  RTA=49  best=0.8240
  random seed=123  (7y9d0k79) ... OK  R*=59  RTA=33  best=0.8424
  random seed=456  (t2cilmtb) ... OK  R*=49  RTA=32  best=0.8370
  fedcs  seed= 42  (44p9iwlr) ... OK  R*=78  RTA=None  best=0.7898
  fedcs  seed=123  (qkdc4s9t) ... OK  R*=55  RTA=None  best=0.7953
  fedcs  seed=456  (ozw36g2z) ... OK  R*=58  RTA=None  best=0.7902
  tifl   seed= 42  (3ius4tsd) ... OK  R*=58  RTA=48  best=0.8234
  tifl   seed=123  (h3do7iw7) ... OK  R*=57  RTA=30  best=0.8336
  tifl   seed=456  (rrvlijkj) ... OK  R*=55  RTA=36  best=0.8330
  oort   seed= 42  (fioevryi) ... OK  R*=41  RTA=31  best=0.8380
  oort   seed=123  (7amp5nx1) ... OK  R*=46  RTA=50  best=0.8294
  oort   seed=456  (x2n1mss6) ... OK  R*=61  RTA=49  best=0.8314

  Alpha = α=2
  random seed= 42  (ukqx8xyv) ... OK  R*=57  RTA=25  best=0.8537
  random seed=123  (jhx7m47m) ... OK  R*=53  RTA=24  best=0.8567
  random seed=456  (teapg762) ... OK  R*=59  RTA=27 

## Summary Helpers

In [7]:
def get_seeds(alpha, strategy):
    """Return list of seeds that have valid data for this (alpha, strategy)."""
    return [
        s for s in all_results[alpha][strategy]
        if all_results[alpha][strategy][s] is not None
    ]


def agg(alpha, strategy, key):
    """Mean ± std of a scalar metric across seeds. Returns (mean, std, n)."""
    vals = [
        all_results[alpha][strategy][s][key]
        for s in get_seeds(alpha, strategy)
        if all_results[alpha][strategy][s].get(key) is not None
    ]
    if not vals:
        return None, None, 0
    return float(np.mean(vals)), float(np.std(vals)), len(vals)


def fmt(mean, std, n, decimals=2):
    """Format mean±std, or just mean if n=1."""
    if mean is None:
        return "—"
    if n <= 1:
        return f"{mean:.{decimals}f}"
    return f"{mean:.{decimals}f}±{std:.{decimals}f}"

## Group 1 Summary (α=2)

In [11]:
def print_group_summary(alpha):
    label = ALPHA_LABELS[alpha]
    seeds_for_alpha = list(RUNS[alpha][STRATEGIES[0]].keys())

    # ── Per-seed table ───────────────────────────────────────────────
    header = (
        f"{'Strategy':<8} {'Seed':>4}  {'R*':>3}  {'RTA':>4}  "
        f"{'TTA(h)':>7}  {'MRD(s)':>7}  {'Acc@R*':>7}  {'BestAcc':>7}  "
        f"{'JFI@R*':>7}  {'JFI@Rm':>7}  {'ΔJFI':>6}  "
        f"{'Uniq':>4}  {'Excl':>4}  {'StragOH':>7}"
    )
    sep = "=" * len(header)

    print(f"\n{'─'*70}")
    print(f"  Group Summary: {label}")
    print(f"{'─'*70}")
    print(f"\n{sep}\n{header}\n{sep}")

    for strategy in STRATEGIES:
        for seed in seeds_for_alpha:
            if seed not in all_results[alpha][strategy]:
                continue
            m = all_results[alpha][strategy][seed]
            if m is None:
                print(f"{strategy:<8} {seed:>4}  MISSING / SKIPPED")
                continue

            rta_s = str(m["rta"]) if m["rta"] else "—"
            tta_s = f"{m['tta']/3600:.2f}" if m["tta"] else "—"
            uniq_s = f"{m['unique_at_r_star']:.0f}" if m["unique_at_r_star"] is not None else "—"
            excl_s = f"{m['never_selected']}" if m["never_selected"] is not None else "—"

            print(
                f"{strategy:<8} {seed:>4}  {m['r_star']:>3}  {rta_s:>4}  "
                f"{tta_s:>7}  {m['mean_round_dur']:>7.1f}  "
                f"{m['acc_at_r_star']:>7.4f}  {m['best_acc']:>7.4f}  "
                f"{m['jfi_at_r_star']:>7.3f}  {m['jfi_at_r_max']:>7.3f}  "
                f"{m['delta_jfi']:>+6.3f}  "
                f"{uniq_s:>4}  {excl_s:>4}  {m['strag_overhead']:>7.3f}"
            )
        print("-" * len(header))

    # ── Seed-aggregated table ────────────────────────────────────────
    print(f"\n── Seed-aggregated (mean ± std) ──")
    agg_header = (
        f"{'Strategy':<8}  {'R*':>10}  {'RTA':>10}  {'TTA(h)':>10}  "
        f"{'MRD(s)':>10}  {'BestAcc':>10}  "
        f"{'JFI@R*':>10}  {'JFI@Rm':>10}  {'ΔJFI':>10}  "
        f"{'NeverSel':>10}"
    )
    print(agg_header)
    print("-" * len(agg_header))

    for strategy in STRATEGIES:
        n = len(get_seeds(alpha, strategy))
        r_m, r_s, _ = agg(alpha, strategy, "r_star")
        rta_m, rta_s, _ = agg(alpha, strategy, "rta")

        # TTA in hours
        tta_vals = [
            all_results[alpha][strategy][s]["tta"] / 3600
            for s in get_seeds(alpha, strategy)
            if all_results[alpha][strategy][s].get("tta") is not None
        ]
        tta_m = np.mean(tta_vals) if tta_vals else None
        tta_s = np.std(tta_vals) if tta_vals else None

        mrd_m, mrd_s, _ = agg(alpha, strategy, "mean_round_dur")
        ba_m, ba_s, _ = agg(alpha, strategy, "best_acc")
        j1_m, j1_s, _ = agg(alpha, strategy, "jfi_at_r_star")
        j2_m, j2_s, _ = agg(alpha, strategy, "jfi_at_r_max")
        dj_m, dj_s, _ = agg(alpha, strategy, "delta_jfi")
        ns_m, ns_s, _ = agg(alpha, strategy, "never_selected")

        print(
            f"{strategy:<8}  {fmt(r_m,r_s,n):>10}  {fmt(rta_m,rta_s,n):>10}  "
            f"{fmt(tta_m,tta_s,n):>10}  {fmt(mrd_m,mrd_s,n,1):>10}  "
            f"{fmt(ba_m,ba_s,n,4):>10}  "
            f"{fmt(j1_m,j1_s,n,3):>10}  {fmt(j2_m,j2_s,n,3):>10}  "
            f"{fmt(dj_m,dj_s,n,3):>10}  {fmt(ns_m,ns_s,n,0):>10}"
        )


# Print Group 1
print_group_summary(2.0)


──────────────────────────────────────────────────────────────────────
  Group Summary: α=2
──────────────────────────────────────────────────────────────────────

Strategy Seed   R*   RTA   TTA(h)   MRD(s)   Acc@R*  BestAcc   JFI@R*   JFI@Rm    ΔJFI  Uniq  Excl  StragOH
random     42   57    25     9.25    478.6   0.8347   0.8537    0.864    0.916  +0.052   300     0    1.981
random    123   53    24     8.72    503.5   0.8390   0.8567    0.844    0.910  +0.066   298     2    1.900
random    456   59    27     9.10    484.6   0.8391   0.8498    0.858    0.910  +0.052   298     2    1.576
-----------------------------------------------------------------------------------------------------------
fedcs      42   61    41     3.65    202.2   0.8005   0.8260    0.211    0.214  +0.003    80   220    0.610
fedcs     123   62    35     3.00    194.8   0.8157   0.8326    0.215    0.216  +0.000    82   218    0.612
fedcs     456   54    33     2.82    197.0   0.8282   0.8370    0.211    0.212 

## Group 2: Cross-Alpha Comparison

In [12]:
def print_cross_alpha():
    """Compare key metrics across heterogeneity levels for each strategy."""

    metrics_to_compare = [
        ("RTA", "rta", 1),
        ("TTA (h)", None, 2),  # special handling: convert to hours
        ("Best Acc", "best_acc", 4),
        ("JFI@R*", "jfi_at_r_star", 3),
        ("ΔJFI", "delta_jfi", 3),
        ("MRD (s)", "mean_round_dur", 1),
        ("Strag OH", "strag_overhead", 3),
        ("Never Sel", "never_selected", 0),
    ]

    print(f"\n{'='*70}")
    print(f"  GROUP 2: Cross-Alpha Comparison")
    print(f"{'='*70}")

    for metric_label, metric_key, decimals in metrics_to_compare:
        alpha_headers = [f"{ALPHA_LABELS[a]:>14}" for a in ALPHAS]
        print(f"\n── {metric_label} ──")
        print(f"{'Strategy':<8}  " + "  ".join(alpha_headers))
        print("-" * (10 + 16 * len(ALPHAS)))

        for strategy in STRATEGIES:
            row = [f"{strategy:<8}"]
            for alpha in ALPHAS:
                n = len(get_seeds(alpha, strategy))
                if n == 0:
                    row.append(f"{'—':>14}")
                    continue

                if metric_key is None and metric_label == "TTA (h)":
                    # Special: convert TTA seconds to hours
                    vals = [
                        all_results[alpha][strategy][s]["tta"] / 3600
                        for s in get_seeds(alpha, strategy)
                        if all_results[alpha][strategy][s] is not None
                        and all_results[alpha][strategy][s].get("tta") is not None
                    ]
                    if not vals:
                        row.append(f"{'—':>14}")
                    elif len(vals) == 1:
                        row.append(f"{vals[0]:>14.{decimals}f}")
                    else:
                        row.append(f"{np.mean(vals):.{decimals}f}±{np.std(vals):.{decimals}f}".rjust(14))
                else:
                    m, s, ct = agg(alpha, strategy, metric_key)
                    row.append(f"{fmt(m, s, ct, decimals):>14}")

            print("  ".join(row))


print_cross_alpha()


  GROUP 2: Cross-Alpha Comparison

── RTA ──
Strategy           α=0.5             α=2             IID
----------------------------------------------------------
random          38.0±7.8        25.3±1.2            28.0
fedcs                  —        36.3±3.4            26.0
tifl            38.0±7.5        28.3±1.9            23.0
oort            43.3±8.7        27.0±0.8            28.0

── TTA (h) ──
Strategy           α=0.5             α=2             IID
----------------------------------------------------------
random        20.69±5.34       9.02±0.22            9.40
fedcs                  —       3.16±0.36            2.50
tifl           7.19±1.31       5.70±1.64            4.01
oort          20.88±2.57       9.79±0.41            8.61

── Best Acc ──
Strategy           α=0.5             α=2             IID
----------------------------------------------------------
random     0.8345±0.0077   0.8534±0.0028          0.8470
fedcs      0.7918±0.0025   0.8319±0.0045          0.8457
tifl 

In [13]:
for alpha in ALPHAS:
    if alpha == 2.0:
        continue  # already printed above
    print_group_summary(alpha)


──────────────────────────────────────────────────────────────────────
  Group Summary: α=0.5
──────────────────────────────────────────────────────────────────────

Strategy Seed   R*   RTA   TTA(h)   MRD(s)   Acc@R*  BestAcc   JFI@R*   JFI@Rm    ΔJFI  Uniq  Excl  StragOH
random     42   59    49    28.23    487.9   0.7686   0.8240    0.870    0.917  +0.047   300     0    3.297
random    123   59    33    16.73    487.1   0.6897   0.8424    0.864    0.910  +0.046   299     1    2.805
random    456   49    32    17.10    494.5   0.7464   0.8370    0.841    0.910  +0.069   297     3    2.746
-----------------------------------------------------------------------------------------------------------
fedcs      42   78     —        —    125.5   0.7834   0.7898    0.216    0.217  +0.001    83   217    0.923
fedcs     123   55     —        —    121.2   0.7630   0.7953    0.214    0.217  +0.003    77   223    0.976
fedcs     456   58     —        —    164.9   0.6887   0.7902    0.217    0.22

## Save

In [14]:
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(all_results, f)
print(f"\nSaved → {OUTPUT_PATH}")
print(f"Structure: all_results[alpha][strategy][seed] → dict of scalars + series")
print(f"Alphas: {list(all_results.keys())}")
for alpha in ALPHAS:
    for strat in STRATEGIES:
        n = len(get_seeds(alpha, strat))
        print(f"  {ALPHA_LABELS[alpha]:>6} / {strat:<8}: {n} seeds")


Saved → results/mnli_all.pkl
Structure: all_results[alpha][strategy][seed] → dict of scalars + series
Alphas: [0.5, 2.0, 'iid']
   α=0.5 / random  : 3 seeds
   α=0.5 / fedcs   : 3 seeds
   α=0.5 / tifl    : 3 seeds
   α=0.5 / oort    : 3 seeds
     α=2 / random  : 3 seeds
     α=2 / fedcs   : 3 seeds
     α=2 / tifl    : 3 seeds
     α=2 / oort    : 3 seeds
     IID / random  : 1 seeds
     IID / fedcs   : 1 seeds
     IID / tifl    : 1 seeds
     IID / oort    : 1 seeds


============== Reconstruct from Raw Table ==================

In [15]:
import json, glob
import pandas as pd

TABLE_COLUMNS = ["round", "node_id", "train_loss", "duration",
                 "compute_time", "communication_time", "num_samples", "selected_by"]

def fetch_history_table(run_id: str) -> pd.DataFrame | None:
    """Reconstruct the incremental client/raw_training_history table.

    Stored as a versioned run artifact (log_mode='INCREMENTAL'). We read every
    .table.json shard across all matching artifacts, concatenate, and
    de-duplicate on (round, node_id). 'node_id' holds the partition/client id.
    """
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
    frames = []
    for art in run.logged_artifacts():
        if "raw_training" not in art.name:
            continue
        try:
            art_dir = art.download()
        except Exception:
            continue
        for shard in glob.glob(f"{art_dir}/**/*.table.json", recursive=True):
            with open(shard) as f:
                tj = json.load(f)
            frames.append(pd.DataFrame(tj["data"], columns=tj.get("columns", TABLE_COLUMNS)))
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    df = df.drop_duplicates(subset=["round", "node_id"]).reset_index(drop=True)
    df["round"] = df["round"].astype(int)
    df["node_id"] = df["node_id"].astype(int)
    return df


def selection_counts(table_df: pd.DataFrame, r_cutoff: int, n_clients: int = N_CLIENTS) -> np.ndarray:
    """Per-client selection count up to round r_cutoff.
    Returns length-n_clients array; clients never selected get 0.
    Each table row is one client-round participation."""
    sub = table_df[table_df["round"] <= r_cutoff]
    counts = sub.groupby("node_id").size()
    full = np.zeros(n_clients, dtype=int)
    for pid, c in counts.items():
        if 0 <= pid < n_clients:
            full[pid] = int(c)
    return full

In [16]:
fetch_history_table('kwo6g2hd')

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


,round,node_id,train_loss,duration,compute_time,communication_time,num_samples,selected_by
0,1,3,0.851446,446.051122,420.445031,25.606091,874,Random(K=30)
1,1,13,1.029025,246.012244,243.360701,2.651543,400,Random(K=30)
2,1,15,1.106486,277.041333,270.901387,6.139946,1041,Random(K=30)
3,1,12,1.245844,245.836280,192.084825,53.751455,487,Random(K=30)
4,1,16,1.057202,266.947390,259.707085,7.240304,693,Random(K=30)
...,...,...,...,...,...,...,...,...
2992,100,268,0.794621,51.965694,33.475363,18.490332,195,Random(K=30)
2993,100,267,0.454614,193.702987,178.614600,15.088387,996,Random(K=30)
2994,100,261,0.165357,1296.777801,1228.730487,68.047314,2629,Random(K=30)
2995,100,290,0.130577,472.834711,450.014573,22.820138,750,Random(K=30)


In [17]:
print("Fetching raw training-history tables...\n")
for alpha in ALPHAS:
    for strategy in STRATEGIES:
        for seed in get_seeds(alpha, strategy):
            m = all_results[alpha][strategy][seed]
            run_id = m["run_id"]
            print(
                f"  {ALPHA_LABELS[alpha]:>6} {strategy:6s} seed={seed:3d} ({run_id}) ...",
                end=" ",
                flush=True,
            )
            try:
                tbl = fetch_history_table(run_id)
                if tbl is None:
                    print("NO TABLE")
                    m["sel_counts_r_star"] = m["sel_counts_r_max"] = None
                    continue
                m["sel_counts_r_star"] = selection_counts(tbl, m["r_star"])
                m["sel_counts_r_max"] = selection_counts(tbl, R_MAX)
                m["history_table"] = tbl  # keep for any later table-based metric
                print(
                    f"OK  rows={len(tbl)}  unique@R*={int((m['sel_counts_r_star']>0).sum())}"
                )
            except Exception as e:
                print(f"FAILED: {e}")
                m["sel_counts_r_star"] = m["sel_counts_r_max"] = None

Fetching raw training-history tables...

   α=0.5 random seed= 42 (kwo6g2hd) ... 

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


OK  rows=2997  unique@R*=300
   α=0.5 random seed=123 (7y9d0k79) ... 

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


OK  rows=3000  unique@R*=299
   α=0.5 random seed=456 (t2cilmtb) ... 

...

OK  rows=3000  unique@R*=80
     IID tifl   seed= 42 (k9g4zmfx) ... 

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


OK  rows=3000  unique@R*=295
     IID oort   seed= 42 (9y2n4bg8) ... 

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   3 of 3 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   5 of 5 files downloaded.  
wandb:   6 of 6 files downloaded.  
wandb:   7 of 7 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   9 of 9 files downloaded.  
wandb:   10 of 10 files downloaded.  
wandb:   11 of 11 files downloaded.  
wandb:   12 of 12 files downloaded.  
wandb:   13 of 13 files downloaded.  
wandb:   14 of 14 files downloaded.  
wandb:   15 of 15 files downloaded.  
wandb:   16 of 16 files downloaded.  
wandb:   17 of 17 files downloaded.  
wandb:   18 of 18 files downloaded.  
wandb:   19 of 19 files downloaded.  
wandb:   20 of 20 files downloaded.  


OK  rows=3000  unique@R*=300


In [18]:
# Sanity Check
alpha = 2.0
print(f"Selection-count distribution at {ALPHA_LABELS[alpha]} (pooled across seeds, up to R*)\n")
print(f"{'Strategy':<8} {'mean':>6} {'std':>6} {'max':>4} {'never':>6}")
print("-" * 36)
for strategy in STRATEGIES:
    pooled = [all_results[alpha][strategy][s]["sel_counts_r_star"]
              for s in get_seeds(alpha, strategy)
              if all_results[alpha][strategy][s].get("sel_counts_r_star") is not None]
    if not pooled:
        continue
    arr = np.concatenate(pooled)
    print(f"{strategy:<8} {arr.mean():>6.2f} {arr.std():>6.2f} {arr.max():>4d} {int((arr==0).sum()):>6d}")

Selection-count distribution at α=2 (pooled across seeds, up to R*)

Strategy   mean    std  max  never
------------------------------------
random     5.63   2.28   14      4
fedcs      5.90  11.37   41    657
tifl       6.10   3.45   16     42
oort       5.93   2.68   13      0


In [23]:
all_results[2.0]["oort"][42]['history_table']

,round,node_id,train_loss,duration,compute_time,communication_time,num_samples,selected_by
0,1,9,1.084054,169.453662,166.369210,3.084452,442,"Oort(K=30,eps=0.90,alpha=2.0)"
1,1,7,1.106393,382.896901,356.298397,26.598504,781,"Oort(K=30,eps=0.90,alpha=2.0)"
2,1,8,1.031940,179.130093,175.164874,3.965219,936,"Oort(K=30,eps=0.90,alpha=2.0)"
3,1,1,1.107931,407.517822,353.448602,54.069220,1158,"Oort(K=30,eps=0.90,alpha=2.0)"
4,1,26,1.039084,550.771895,464.403785,86.368109,2028,"Oort(K=30,eps=0.90,alpha=2.0)"
...,...,...,...,...,...,...,...,...
2992,100,260,0.312837,250.308236,199.461157,50.847079,1192,"Oort(K=30,eps=0.20,alpha=2.0)"
2993,100,267,0.341592,260.952460,245.864073,15.088387,1371,"Oort(K=30,eps=0.20,alpha=2.0)"
2994,100,281,0.447956,210.123494,108.816268,101.307225,743,"Oort(K=30,eps=0.20,alpha=2.0)"
2995,100,288,0.442321,132.669061,111.103874,21.565187,349,"Oort(K=30,eps=0.20,alpha=2.0)"


In [19]:
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(all_results, f)
print(f"Updated → {OUTPUT_PATH} (added sel_counts_r_star, sel_counts_r_max, history_table)")

Updated → results/mnli_all.pkl (added sel_counts_r_star, sel_counts_r_max, history_table)


## Statistical Testing: Paired t-test vs Random

For each metric, compute a paired (one-sample) t-test on the per-seed
differences (Random − strategy). Pairing is valid because the same seeds
run across all strategies. IID has one seed, so it is skipped (n < 2).

In [4]:
with open("results/mnli_all.pkl", "rb") as f:
    all_results = pickle.load(f)

print("Loaded keys:", {str(a): list(all_results[a].keys()) for a in all_results})

Loaded keys: {'0.5': ['random', 'fedcs', 'tifl', 'oort'], '2.0': ['random', 'fedcs', 'tifl', 'oort'], 'iid': ['random', 'fedcs', 'tifl', 'oort']}


In [5]:
from scipy import stats  # used only for the t-distribution p-value lookup

# metric key in all_results -> display name
METRICS_FOR_TEST = {
    "r_star":         "R*",
    "rta":            "RTA",
    "tta":            "TTA",        # stored in seconds; units do not affect p
    "mean_round_dur": "MRD",
    "best_acc":       "BestAcc",
    "acc_at_r_max":   "Acc@Rmax",
    "jfi_at_r_star":  "JFI@R*",
    "strag_overhead": "StragOH",
}


def paired_diffs(alpha, strategy, key, reference="random"):
    """Per-seed differences (reference - strategy), paired on common seeds."""
    seeds = [
        s for s in get_seeds(alpha, reference)
        if s in all_results[alpha][strategy]
        and all_results[alpha][strategy][s] is not None
        and all_results[alpha][reference][s].get(key) is not None
        and all_results[alpha][strategy][s].get(key) is not None
    ]
    diffs = [
        all_results[alpha][reference][s][key] - all_results[alpha][strategy][s][key]
        for s in seeds
    ]
    return np.array(diffs, dtype=float), seeds


def paired_ttest(alpha, strategy, key, reference="random"):
    """Manual paired t-test. Returns the full arithmetic, not just p."""
    d, seeds = paired_diffs(alpha, strategy, key, reference)
    n = len(d)
    if n < 2:
        return None

    mean_d = d.mean()                 # step 2: mean of differences
    sd_d   = d.std(ddof=1)            # step 3: sample std (divide by n-1)
    se     = sd_d / np.sqrt(n)        # step 4: standard error
    df     = n - 1                    # degrees of freedom

    if se == 0:                       # all differences identical
        t = 0.0 if mean_d == 0 else np.inf
    else:
        t = mean_d / se               # step 5: t-statistic

    p = 2 * stats.t.sf(abs(t), df)    # step 7: two-tailed p-value

    return {"n": n, "diffs": d, "mean_d": mean_d, "sd_d": sd_d,
            "se": se, "df": df, "t": t, "p": p}

In [8]:
def print_ttests(alpha, reference="random"):
    label = ALPHA_LABELS[alpha]
    print(f"\nPaired t-test vs {reference} — alpha = {label}")
    print(f"{'Strategy':<8} {'Metric':<9} {'n':>2} {'mean_d':>11} "
          f"{'sd_d':>9} {'t':>8} {'p':>8}  sig")
    print("-" * 66)
    for strategy in STRATEGIES:
        if strategy == reference:
            continue
        for key, name in METRICS_FOR_TEST.items():
            r = paired_ttest(alpha, strategy, key, reference)
            if r is None:
                print(f"{strategy:<8} {name:<9}  (n<2, skipped)")
                continue
            p = r["p"]
            sig = "**" if p < 0.01 else "*" if p < 0.05 else "." if p < 0.10 else "ns"
            t_s = "inf" if np.isinf(r["t"]) else f"{r['t']:.2f}"
            print(f"{strategy:<8} {name:<9} {r['n']:>2} {r['mean_d']:>+11.4f} "
                  f"{r['sd_d']:>9.4f} {t_s:>8} {p:>8.4f}  {sig}")
        print("-" * 66)


print_ttests(2.0)
print_ttests(0.5)   # FedCS RTA/TTA auto-skip (None); IID would skip on n<2


Paired t-test vs random — alpha = α=2
Strategy Metric     n      mean_d      sd_d        t        p  sig
------------------------------------------------------------------
fedcs    R*         3     -2.6667    7.0946    -0.65   0.5818  ns
fedcs    RTA        3    -11.0000    5.0000    -3.81   0.0625  .
fedcs    TTA        3 +21121.3947 1306.0976    28.01   0.0013  **
fedcs    MRD        3   +290.8616   16.3626    30.79   0.0011  **
fedcs    BestAcc    3     +0.0216    0.0078     4.81   0.0406  *
fedcs    Acc@Rmax   3     +0.0217    0.0053     7.14   0.0191  *
fedcs    JFI@R*     3     +0.6433    0.0128    86.79   0.0001  **
fedcs    StragOH    3     +1.2079    0.2155     9.71   0.0104  *
------------------------------------------------------------------
tifl     R*         3     -4.6667    1.5275    -5.29   0.0339  *
tifl     RTA        3     -3.0000    1.0000    -5.20   0.0351  *
tifl     TTA        3 +11967.5498 8230.4078     2.52   0.1281  ns
tifl     MRD        3     +3.9502   38.3